# SSB Oppdateringsdetector – Complete Version

Kjøres periodisk (f.eks. daglig) fra en Fabric Pipeline.
Leser `ssb_config`, sjekker SSB API for oppdateringer, og skriver output-tabeller.

## Output

| Tabell | Innhold |
|---|---|
| `statbank_staging.pipeline.ssb_load_queue_critical` | Kun tabeller med `priority = CRITICAL` |
| `statbank_staging.pipeline.ssb_load_queue` | Alle tabeller (CRITICAL + NORMAL) |
| `statbank_staging.pipeline.ssb_update_log` | Audit-log per kjøring |
| *(exit-verdi)* | Samme informasjon som JSON, levert til Fabric Pipeline via `notebook.exit()` – ikke en fil |

## Logikk
- **first_load**: tabeller uten `last_loaded_timestamp` → alltid lastet
- **updated**: tabeller der SSB sin `updated`-timestamp er nyere enn `last_loaded_timestamp`
- **DYNAMIC_PAST_DAYS**: beregnes automatisk fra siste vellykkede audit-logg, med 1 dags buffer

---------

In [ ]:
# ============================================================================
# IMPORTS OG KONFIGURASJON
# ============================================================================
# Kobler til Spark og setter grunninnstillingene for denne kjøringen: hvor
# ssb_config ligger, hvor mange dager bakover man faller tilbake til hvis
# audit-loggen mangler, og et tidsstempel (CHECK_TIMESTAMP) som merker alt
# denne kjøringen produserer – slik at 04 og 05 senere kan se hvilken
# kjøring av denne notebooken som ligger bak dataene de jobber med.

import json
import time
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone
from typing import Dict, List, Optional, Tuple
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType
)
from delta.tables import DeltaTable
from notebookutils import mssparkutils

spark = SparkSession.builder.appName("SSBOppdateringsdetector").getOrCreate()

# ===== PARAMETERE =====
CONFIG_TABLE     = "statbank_staging.pipeline.ssb_config"
FALLBACK_DAYS    = 7       # brukes hvis audit-logg ikke finnes
MAX_SSB_DAYS     = 90      # øvre grense for pastdays mot SSB API
SSB_PAGESIZE     = 100     # antall tabeller per side fra SSB API
CHECK_TIMESTAMP  = datetime.now(timezone.utc).isoformat()

print(f"🚀 SSB Oppdateringsdetector")
print(f"   Config-tabell:  {CONFIG_TABLE}")
print(f"   Fallback days:  {FALLBACK_DAYS}")
print(f"   Check-tid:      {CHECK_TIMESTAMP}")


# ============================================================================
# SCHEMAS
# ============================================================================
# Definerer den faste strukturen på de to tabellene denne notebooken
# skriver: køen med tabeller som skal lastes (UPDATED_TABLES_SCHEMA), og
# logg-raden som lagres for hver kjøring (AUDIT_LOG_SCHEMA).

UPDATED_TABLES_SCHEMA = StructType([
    StructField("table_id",               StringType(),  False),
    StructField("config_name",            StringType(),  False),
    StructField("frequency",              StringType(),  False),
    StructField("lookback_periods",       IntegerType(), False),
    StructField("reason",                 StringType(),  False),
    StructField("check_timestamp",        StringType(),  False),
    StructField("last_downloaded_timestamp",  StringType(),  True),
    StructField("ssb_updated_timestamp",  StringType(),  True),
    StructField("priority",               StringType(),  False),
])

AUDIT_LOG_SCHEMA = StructType([
    StructField("check_timestamp",        StringType(),  False),
    StructField("config_tables_count",    IntegerType(), False),
    StructField("ssb_updated_count",      IntegerType(), False),
    StructField("first_load_count",       IntegerType(), False),
    StructField("updated_count",          IntegerType(), False),
    StructField("total_to_load",          IntegerType(), False),
    StructField("past_days_checked",      IntegerType(), False),   # DYNAMIC_PAST_DAYS
    StructField("status",                 StringType(),  False),
])

print("✅ Imports og schemas OK")

In [ ]:
# ============================================================================
# FUNKSJONER
# ============================================================================
# Alt "tankearbeidet" i notebooken samlet: lese hvilke tabeller som følges
# (load_config_from_lakehouse), regne ut hvor mange dager bakover man bør
# spørre SSB om (get_dynamic_past_days), hente listen over SSBs egne
# oppdateringer (fetch_updated_tables_from_ssb), og sammenligne de to for å
# avgjøre hvilke tabeller som faktisk trenger en ny nedlasting
# (cross_reference_tables). De nummererte stegene lenger ned kaller disse i
# rekkefølge.

def _ensure_config_columns():
    """
    Migrering: legg til priority- og last_downloaded_timestamp-kolonner i
    ssb_config hvis de mangler. Trygg å kalle gjentatte ganger.
    """
    try:
        existing = [f.name for f in spark.table(CONFIG_TABLE).schema.fields]
        if "priority" not in existing:
            print("⚙️  Legger til 'priority'-kolonne i ssb_config...")
            spark.sql(f"ALTER TABLE {CONFIG_TABLE} ADD COLUMN priority STRING")
            spark.sql(f"UPDATE {CONFIG_TABLE} SET priority = 'NORMAL' WHERE priority IS NULL")
            print("✅ 'priority' lagt til (alle satt til NORMAL)")
        if "last_downloaded_timestamp" not in existing:
            print("⚙️  Legger til 'last_downloaded_timestamp'-kolonne i ssb_config...")
            spark.sql(f"ALTER TABLE {CONFIG_TABLE} ADD COLUMN last_downloaded_timestamp STRING")
            print("✅ 'last_downloaded_timestamp' lagt til")
    except Exception:
        pass  # tabellen finnes kanskje ikke ennå


def load_config_from_lakehouse(table_name: str = CONFIG_TABLE) -> List[Dict]:
    """Les ssb_config fra Delta-tabell og returner som liste med dicts."""
    print(f"\n📋 Leser config: {table_name}")
    _ensure_config_columns()

    config_df   = spark.table(table_name)
    config_rows = config_df.collect()

    if not config_rows:
        raise ValueError(f"Config-tabell '{table_name}' er tom!")

    tables = []
    for row in config_rows:
        row_dict = row.asDict()
        tables.append({
            "id":                   row_dict["table_id"],
            "name":                 row_dict["table_name"],
            "frequency":            row_dict["frequency"] or "Unknown",
            "category":             row_dict.get("category") or "",
            "priority":             (row_dict.get("priority") or "NORMAL").upper(),
            "lookback_periods":     int(row_dict["lookback_periods"]) if row_dict["lookback_periods"] is not None else 2,
            "last_downloaded_timestamp": row_dict.get("last_downloaded_timestamp"),
        })

    print(f"✅ Lastet {len(tables)} tabeller fra config")
    for t in tables[:5]:
        status = "🆕 first load" if not t["last_downloaded_timestamp"] else f"✅ {t['last_downloaded_timestamp'][:19]}"
        print(f"   [{t['priority']:8}] {t['id']}: {t['name'][:40]:<40} ({status})")
    if len(tables) > 5:
        print(f"   ... og {len(tables) - 5} flere")

    return tables


def get_dynamic_past_days(fallback_days: int = FALLBACK_DAYS) -> int:
    """
    Beregn antall dager å søke bakover basert på siste vellykkede kjøring.
    Legger til 1 dags buffer. Capper på MAX_SSB_DAYS.
    Faller tilbake til fallback_days hvis audit-logg ikke finnes.
    """
    try:
        last_run_row = spark.sql("""
            SELECT MAX(CAST(check_timestamp AS TIMESTAMP)) AS last_run
            FROM statbank_staging.pipeline.ssb_update_log
            WHERE status IN ('success', 'no_updates')
        """).collect()[0]

        last_run = last_run_row["last_run"]
        if last_run is not None:
            # last_run kan være timezone-aware (fra audit-log med UTC Z)
            now = datetime.now(timezone.utc)
            if last_run.tzinfo is None:
                last_run = last_run.replace(tzinfo=timezone.utc)
            gap_days = (now - last_run).days + 1
            dynamic = max(gap_days, 1)
            dynamic = min(dynamic, MAX_SSB_DAYS)
            print(f"📅 Siste vellykkede kjøring: {last_run.isoformat()[:19]}")
            print(f"🔄 Beregnet PAST_DAYS: {dynamic} (gap={gap_days-1}d + 1d buffer, cap={MAX_SSB_DAYS})")
            return dynamic
        else:
            print(f"⚠️  Ingen tidligere logg funnet. Bruker fallback: {fallback_days} dager")
            return fallback_days
    except Exception as e:
        print(f"ℹ️  Audit-logg utilgjengelig ({e}). Bruker fallback: {fallback_days} dager")
        return fallback_days


def fetch_updated_tables_from_ssb(past_days: int) -> List[Dict]:
    """
    Hent alle tabeller oppdatert siste past_days dager fra SSB API.
    Håndterer paginering og retry med backoff.
    """
    base_url    = "https://data.ssb.no/api/pxwebapi/v2/tables"
    all_tables: List[Dict] = []
    page_number = 1
    total_pages = None

    while True:
        params = {
            "lang":       "no",
            "pagesize":   SSB_PAGESIZE,
            "pagenumber": page_number,
            "pastdays":   past_days,
        }
        last_err: Exception = RuntimeError("Ingen forsøk gjort")
        for attempt in range(1, 4):
            try:
                resp = requests.get(
                    base_url, params=params, timeout=30,
                    headers={
                        "User-Agent":       "FabricDataPipeline/3.0",
                        "Accept":           "application/json",
                        "Accept-Language":  "no",
                    },
                )
                resp.raise_for_status()
                data = resp.json()

                batch = data.get("tables", [])
                all_tables.extend(batch)

                page_info   = data.get("page", {})
                total_pages = page_info.get("totalPages", 1)
                print(f"   Side {page_number}/{total_pages}: {len(batch)} tabeller (total hittil: {len(all_tables)})")
                break
            except requests.RequestException as e:
                last_err = e
                print(f"   ⚠️  Side {page_number}, forsøk {attempt}/3 feilet: {e}")
                if attempt < 3:
                    time.sleep(3 * attempt)
                else:
                    raise RuntimeError(
                        f"SSB API utilgjengelig etter 3 forsøk: {last_err}"
                    ) from last_err

        if total_pages is not None and page_number >= total_pages:
            break
        page_number += 1

    print(f"✅ Totalt {len(all_tables)} tabeller hentet fra SSB")
    return all_tables


def _parse_dt(dt_string: Optional[str]) -> Optional[datetime]:
    """Robust datetime-parsing med timezone-normalisering til UTC."""
    if not dt_string:
        return None
    for fmt in (
        "%Y-%m-%dT%H:%M:%S%z",
        "%Y-%m-%dT%H:%M:%S.%f%z",
        "%Y-%m-%dT%H:%M:%S",
        "%Y-%m-%dT%H:%M:%S.%f",
    ):
        try:
            dt = datetime.strptime(dt_string.replace("Z", "+00:00"), fmt)
            if dt.tzinfo is None:
                dt = dt.replace(tzinfo=timezone.utc)
            return dt
        except ValueError:
            continue
    return None


def cross_reference_tables(
    config_tables: List[Dict],
    ssb_updated_tables: List[Dict],
) -> pd.DataFrame:
    """
    Kryss-referanse config mot SSB-oppdateringer.
    Returnerer DataFrame med tabeller som skal lastes.
    """
    ssb_by_id = {t["id"]: t for t in ssb_updated_tables}

    to_load: List[Dict] = []
    first_load_count = updated_count = 0

    print("\n🔍 Analyserer tabeller...")

    for ct in config_tables:
        table_id    = ct["id"]
        last_loaded = ct.get("last_downloaded_timestamp")

        # Normaliser: tom streng = None
        if last_loaded == "":
            last_loaded = None

        base_row = {
            "table_id":              str(table_id),
            "config_name":           str(ct["name"]),
            "frequency":             str(ct["frequency"]),
            "lookback_periods":      int(ct.get("lookback_periods", 2)),
            "check_timestamp":       CHECK_TIMESTAMP,
            "last_downloaded_timestamp": last_loaded or "",
            "ssb_updated_timestamp": "",
        }

        # CASE 1: First load
        if not last_loaded:
            print(f"   🆕 {table_id}: first_load")
            to_load.append({**base_row, "reason": "first_load"})
            first_load_count += 1
            continue

        # CASE 2: SSB har oppdatert etter siste last
        if table_id in ssb_by_id:
            ssb_updated_ts = _parse_dt(ssb_by_id[table_id].get("updated"))
            last_loaded_dt = _parse_dt(last_loaded)

            if ssb_updated_ts and last_loaded_dt and ssb_updated_ts > last_loaded_dt:
                print(f"   ✅ {table_id}: updated ({ssb_updated_ts.isoformat()[:19]} > {last_loaded_dt.isoformat()[:19]})")
                to_load.append({
                    **base_row,
                    "reason":                "updated",
                    "ssb_updated_timestamp": ssb_updated_ts.isoformat(),
                })
                updated_count += 1

    result_df = pd.DataFrame(to_load) if to_load else pd.DataFrame(
        columns=[
            "table_id", "config_name", "frequency", "lookback_periods",
            "reason", "check_timestamp", "last_downloaded_timestamp", "ssb_updated_timestamp", "priority",
        ]
    )

    print(f"\n📊 Resultat:")
    print(f"   Config:      {len(config_tables)} tabeller")
    print(f"   SSB oppdaterte (siste {DYNAMIC_PAST_DAYS}d): {len(ssb_updated_tables)}")
    print(f"   First load:  {first_load_count}")
    print(f"   Oppdatert:   {updated_count}")
    print(f"   TOTALT:      {len(to_load)}")

    return result_df


def _overwrite_output_table(
    pdf: pd.DataFrame,
    table_name: str,
    check_ts: str,
):
    """Erstatt innholdet fullstendig ved hver kjøring – idempotent."""
    sdf = spark.createDataFrame(pdf, schema=UPDATED_TABLES_SCHEMA)
    (
        sdf.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )
    print(f"   {table_name} overskrevet ({len(pdf)} rader)")

In [ ]:
# ============================================================================
# STEG 1: Les config
# ============================================================================
# Henter listen over alle SSB-tabeller pipelinen skal følge med på, og når
# hver av dem sist ble hentet ned (fra ssb_config).

print("\n" + "="*70)
print("STEG 1: Les config")
print("="*70)

config_tables = load_config_from_lakehouse(CONFIG_TABLE)

In [ ]:
# ============================================================================
# STEG 2: Dynamisk PAST_DAYS og hent fra SSB API
# ============================================================================
# Spør SSB: "hvilke tabeller har dere publisert nye tall for i det siste?"
# Hvor langt tilbake i tid man spør avhenger av når pipelinen sist kjørte
# vellykket – har det gått lenge (f.eks. en helg, eller en feilet kjøring),
# utvides søket automatisk slik at ingenting glipper mellom kjøringene.

print("\n" + "="*70)
print("STEG 2: Dynamisk henting fra SSB API")
print("="*70)

DYNAMIC_PAST_DAYS = get_dynamic_past_days(FALLBACK_DAYS)

print(f"\n📡 Henter SSB-tabeller oppdatert siste {DYNAMIC_PAST_DAYS} dager...")
ssb_updated = fetch_updated_tables_from_ssb(DYNAMIC_PAST_DAYS)

In [ ]:
# ============================================================================
# STEG 3: Kryss-referanse
# ============================================================================
# Sammenligner de to listene fra steg 1 og 2: for hver tabell vi følger med
# på, sjekker vi om den enten aldri har vært lastet før (first_load), eller
# om SSB har publisert noe nyere enn det vi sist hentet (updated). Tabeller
# uten endringer filtreres bort her – de skal ikke lastes på nytt.

print("\n" + "="*70)
print("STEG 3: Kryss-referanse")
print("="*70)

updated_df = cross_reference_tables(config_tables, ssb_updated)

In [ ]:
# ============================================================================
# STEG 4: Hent priority og splitt i CRITICAL / ALL
# ============================================================================
# Tabellene som trenger oppdatering (fra steg 3) skrives til to køer: én med
# absolutt alle, og én med kun de som er merket CRITICAL i ssb_config. Dette
# lar en Fabric Pipeline velge å prioritere de kritiske tabellene i en egen,
# raskere kjøring hvis ønskelig.

print("\n" + "="*70)
print("STEG 4: Splitt i CRITICAL og ALL")
print("="*70)

has_updates = len(updated_df) > 0
critical_df       = pd.DataFrame()
updated_with_prio = pd.DataFrame()

if has_updates:
    # Bygg priority-mapping fra config_tables (allerede lastet – ingen ekstra Spark-query)
    prio_map = {ct["id"]: ct["priority"] for ct in config_tables}
    updated_with_prio = updated_df.copy()
    updated_with_prio["priority"] = (
        updated_with_prio["table_id"]
        .map(prio_map)
        .fillna("NORMAL")
        .str.upper()
    )

    critical_df = updated_with_prio[updated_with_prio["priority"] == "CRITICAL"].copy()
    print(f"   CRITICAL: {len(critical_df)} tabeller")
    print(f"   NORMAL:   {len(updated_with_prio) - len(critical_df)} tabeller")

    critical_to_save = critical_df
    _overwrite_output_table(
        critical_to_save,
        "statbank_staging.pipeline.ssb_load_queue_critical",
        CHECK_TIMESTAMP,
    )


    # LAGRE ALL
    all_to_save = updated_with_prio
    _overwrite_output_table(
        all_to_save,
        "statbank_staging.pipeline.ssb_load_queue",
        CHECK_TIMESTAMP,
    )

else:
    print("   ℹ️  Ingen tabeller å laste – output-tabeller ikke endret")


# ============================================================================
# Audit-log  (NB: bruker DYNAMIC_PAST_DAYS – ikke FALLBACK_DAYS)
# ============================================================================
# Skriver én rad i historikk-loggen for denne kjøringen – hvor mange
# tabeller ble sjekket, hvor mange trengte oppdatering, og om kjøringen gikk
# bra. Denne loggen er det steg 2 leser fra neste gang, for å vite hvor
# langt tilbake i tid den bør spørre SSB.
print("\n💾 Lagrer audit-log...")

fl_count = int((updated_df["reason"] == "first_load").sum()) if has_updates else 0
up_count = int((updated_df["reason"] == "updated").sum())    if has_updates else 0

audit_row = [{
    "check_timestamp":     CHECK_TIMESTAMP,
    "config_tables_count": len(config_tables),
    "ssb_updated_count":   len(ssb_updated),
    "first_load_count":    fl_count,
    "updated_count":       up_count,
    "total_to_load":       len(updated_df),
    "past_days_checked":   DYNAMIC_PAST_DAYS,   # ← korrekt verdi
    "status":              "success" if has_updates else "no_updates",
}]

audit_sdf = spark.createDataFrame(audit_row, schema=AUDIT_LOG_SCHEMA)
# Viktig: skriv audit-loggen eksplisitt til samme lakehouse (statbank) og schema (pipeline)
audit_sdf.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("statbank_staging.pipeline.ssb_update_log")

print("✅ Audit-log lagret")


# ============================================================================
# Vis sammendrag
# ============================================================================
# Ren visning – lister opp alle tabellene som ble lagt i køen denne
# kjøringen, med prioritet og grunn (first_load/updated).
print("\n📋 Tabeller som skal lastes:")
if has_updates:
    for _, row in updated_with_prio.iterrows():
        print(
            f"   [{row['priority']:8}] {row['table_id']}: "
            f"{row['config_name'][:35]:<35} – {row['reason']}"
        )
else:
    print("   (Ingen tabeller)")

In [ ]:
# ============================================================================
# STEG 5: Pipeline output JSON
# ============================================================================
# Pakker sammen resultatet av hele kjøringen (hvilke tabeller, hvorfor, og
# litt statistikk) til én JSON-pakke, og leverer den tilbake til Fabric
# Pipeline via notebook.exit() helt til slutt. Selve dataene tabellene 04
# og 05 trenger ligger uansett trygt i ssb_load_queue-tabellene – dette er
# først og fremst til logging og eventuell videre bruk i pipelinen.

print("\n" + "="*70)
print("STEG 5: Pipeline output")
print("="*70)

table_ids = updated_df["table_id"].tolist() if has_updates else []

tables_detailed = [
    {
        "table_id":        str(row["table_id"]),
        "config_name":     str(row["config_name"]),
        "frequency":       str(row["frequency"]),
        "lookback_periods": int(row["lookback_periods"]),
        "reason":          str(row["reason"]),
        "priority":        str(updated_with_prio.loc[idx, "priority"]) if has_updates else "NORMAL",
    }
    for idx, row in updated_df.iterrows()
]

output_config = {
    "check_timestamp":  CHECK_TIMESTAMP,
    "from_date":        (
        datetime.now(timezone.utc) - timedelta(days=DYNAMIC_PAST_DAYS)
    ).isoformat(),
    "past_days_checked": DYNAMIC_PAST_DAYS,
    "table_count":      int(len(table_ids)),
    "table_ids":        table_ids,
    "tables_detailed":  tables_detailed,
    "audit": {
        "config_tables_count": len(config_tables),
        "critical_count":      int(len(critical_df)),
        "normal_count":        int(len(updated_with_prio)) - int(len(critical_df)) if has_updates else 0,
        "first_load_count":    fl_count,
        "updated_count":       up_count,
    },
}

print(f"\n✅ Ferdig! {len(table_ids)} tabeller klare")
if table_ids:
    print(f"   IDs: {', '.join(table_ids)}")

output_json_str = json.dumps(output_config, indent=2, ensure_ascii=False)
print(f"\n📦 Output JSON:")
print(output_json_str)

print("\n" + "="*70)
print("✅ SSB UPDATE DETECTOR FULLFØRT")
print("="*70)

# Gjør output tilgjengelig for kallende Fabric Pipeline via notebook-exit-verdien
# (tidligere ble dette skrevet til /tmp/pipeline.ssb_load_queue.json på driver-noden,
# som ikke overlever til neste pipeline-aktivitet og aldri ble lest av noe)
mssparkutils.notebook.exit(output_json_str)